# Assignment 6 - Deep Feelings

This notebook builds sentiment classification systems for the tweet data in `A06_train.csv` and `A06_test.csv`.

The labels are:

- `-1`: negative
- `0`: neutral
- `1`: positive

## Step 0: Install Required Packages

Run this cell first if the notebook environment does not already have the required libraries installed.

In [ ]:
import sys

!{sys.executable} -m pip install scikit-learn

## Step 1: Import Libraries

This section imports the libraries used for reading the CSV files, building the bag-of-words features, training the classifier, and measuring accuracy.

In [ ]:
import csv
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

## Step 2: Load the Training and Test Data

Each row in the CSV files contains a sentiment label in the first column and the tweet text in the remaining columns. A CSV reader is used so that commas inside tweets are handled correctly.

In [ ]:
def load_dataset(path):
    labels = []
    texts = []
    with open(path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        for row in reader:
            labels.append(int(row[0]))
            texts.append(",".join(row[1:]).strip())
    return labels, texts

train_labels, train_texts = load_dataset("A06_train.csv")
test_labels, test_texts = load_dataset("A06_test.csv")

print("Training examples:", len(train_texts))
print("Test examples:", len(test_texts))

## Step 3: Inspect the Label Distribution

Before training the models, it helps to check how the labels are distributed in the training and test sets.

In [ ]:
print("Training label counts:", Counter(train_labels))
print("Test label counts:", Counter(test_labels))

The dataset sizes used in my run were:

- Training examples: `26499`
- Test examples: `6625`

Training label distribution:

- Negative (`-1`): `5368`
- Neutral (`0`): `11510`
- Positive (`1`): `9621`

Test label distribution:

- Negative (`-1`): `1319`
- Neutral (`0`): `2877`
- Positive (`1`): `2429`

## Step 4: Create a Helper Function for Experiments

All of the bag-of-words systems use the same classifier: `LogisticRegression`. The only thing that changes between experiments is how the `CountVectorizer` is configured.

The helper function below trains a pipeline, predicts on the test set, and returns the accuracy and detailed evaluation.

In [ ]:
def run_experiment(name, **vectorizer_kwargs):
    pipeline = Pipeline([
        ("vectorizer", CountVectorizer(**vectorizer_kwargs)),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ])
    pipeline.fit(train_texts, train_labels)
    predictions = pipeline.predict(test_texts)
    accuracy = accuracy_score(test_labels, predictions)
    return {
        "name": name,
        "pipeline": pipeline,
        "predictions": predictions,
        "accuracy": accuracy,
        "report": classification_report(test_labels, predictions, digits=4),
        "confusion": confusion_matrix(test_labels, predictions, labels=[-1, 0, 1]),
    }

## Step 5: Train the Baseline System

The baseline system uses a plain bag-of-words representation with `CountVectorizer` unigram features. This gives a reference point for comparing the enhanced systems.

In [ ]:
baseline_result = run_experiment("baseline_bow")
print("Baseline accuracy:", baseline_result["accuracy"])
print(baseline_result["report"])

In my run, the baseline accuracy was `0.6475`, which is close to the assignment expectation of about `65%`.

## Step 6: Train the Enhanced Bag-of-Words Systems

The assignment requires testing enhancements to the baseline. I used these feature changes:

- stopword removal
- bigram features
- stopword removal together with bigram features

In [ ]:
stopwords_result = run_experiment("stopwords_removed", stop_words="english")
bigrams_result = run_experiment("bigrams_added", ngram_range=(1, 2))
stopwords_bigrams_result = run_experiment(
    "stopwords_plus_bigrams",
    stop_words="english",
    ngram_range=(1, 2),
)

results = [baseline_result, stopwords_result, bigrams_result, stopwords_bigrams_result]
for result in results:
    print(f"{result['name']}: {result['accuracy']:.4f}")

## Step 7: Compare the Accuracy of All Bag-of-Words Systems

This comparison shows whether each enhancement improved over the baseline.

In [ ]:
baseline_accuracy = baseline_result["accuracy"]

for result in results:
    delta = result["accuracy"] - baseline_accuracy
    print(f"{result['name']}: accuracy={result['accuracy']:.4f}, change={delta:+.4f}")

The results from my run were:

| System | Accuracy | Change vs. baseline |
|---|---:|---:|
| Baseline bag-of-words | `0.6475` | `0.0000` |
| Stopwords removed | `0.6411` | `-0.0065` |
| Bigrams added | `0.6562` | `+0.0086` |
| Stopwords removed + bigrams | `0.6614` | `+0.0139` |

## Step 8: Inspect the Best Model

The best-performing bag-of-words model in my run was the system that used both stopword removal and bigrams.

In [ ]:
best_result = max(results, key=lambda result: result["accuracy"])
print("Best model:", best_result["name"])
print("Accuracy:", best_result["accuracy"])
print(best_result["report"])
print(best_result["confusion"])

In my run, the best model was `stopwords_plus_bigrams` with `0.6614` accuracy.

Its classification report was:

```text
              precision    recall  f1-score   support

          -1     0.6118    0.5519    0.5803      1319
           0     0.6390    0.6934    0.6651      2877
           1     0.7173    0.6830    0.6997      2429

    accuracy                         0.6614      6625
   macro avg     0.6560    0.6428    0.6484      6625
weighted avg     0.6623    0.6614    0.6609      6625
```

## Step 9: Optional Embeddings Experiment

The assignment also asks for an embeddings-based system using spaCy. The code below attempts that experiment only if spaCy and a vector model are installed. If they are not installed, the notebook skips the experiment instead of crashing.

If you want to run the embeddings experiment, install both the spaCy library and the English medium model first.

In [ ]:
import sys

!{sys.executable} -m pip install spacy
!{sys.executable} -m spacy download en_core_web_md

In [ ]:
try:
    import spacy
    import numpy as np
    from sklearn.base import BaseEstimator, TransformerMixin

    class SpacyVectorizer(BaseEstimator, TransformerMixin):
        def __init__(self, model_name="en_core_web_md"):
            self.model_name = model_name
            self.nlp = None

        def fit(self, X, y=None):
            self.nlp = spacy.load(self.model_name, disable=["parser", "ner", "textcat"])
            if self.nlp.vocab.vectors_length == 0:
                raise ValueError("The spaCy model does not include vectors.")
            return self

        def transform(self, X):
            docs = self.nlp.pipe(X, batch_size=256)
            return np.vstack([doc.vector for doc in docs])

    embeddings_pipeline = Pipeline([
        ("vectors", SpacyVectorizer()),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ])

    embeddings_pipeline.fit(train_texts, train_labels)
    embeddings_predictions = embeddings_pipeline.predict(test_texts)
    embeddings_accuracy = accuracy_score(test_labels, embeddings_predictions)
    print("spaCy embeddings accuracy:", embeddings_accuracy)
except Exception as exc:
    print("spaCy embeddings experiment not run:", exc)

In my environment, the spaCy embeddings experiment did not run because spaCy was not installed. Because of that, I cannot make a valid performance comparison for the embeddings model from this run.

## Step 10: Analysis

### Enhancement That Did Not Improve Performance

Removing stopwords alone did not improve the model. Accuracy dropped from `0.6475` to `0.6411`, which is a decrease of `0.0065`.

A likely reason is that tweets are short, so common words can still carry useful context or sentiment. Some short texts depend heavily on small words and negations to express tone. If those words are removed, the model loses information that helps distinguish negative, neutral, and positive tweets.

One possible way to improve this version would be to use a custom stopword list instead of the default English list. That would allow important sentiment words such as negations to remain in the text. Another possible improvement would be to add lemmatization or more tweet-specific preprocessing.

### Enhancements That Did Improve Performance

Adding bigrams improved accuracy from `0.6475` to `0.6562`, for an increase of `0.0086`.

Using both stopword removal and bigrams improved accuracy from `0.6475` to `0.6614`, for an increase of `0.0139`.

These improvements make sense because bigrams capture short phrase patterns that unigrams cannot. In tweets, sentiment is often expressed by short combinations of words rather than by individual words alone.

The increase may not come only from the enhancement idea itself. Part of the improvement may also be due to the larger and more specific feature space created by adding bigrams.

### Additional Observation

The best model still confused some sentiment-heavy tweets with neutral tweets. This suggests that the neutral class is harder to separate from weakly positive or weakly negative language. The classification report also shows that the model performed best on positive tweets and worse on negative tweets.